# Day 16 — Filtering, Selection & Indexing
### Python for Data Science · Module 1 · Topic 1.15

**Prepared & presented by Srinivasa Sai Chava**  ·  Boston University

---

**Session length:** 2 hours
**Format:** 90 min concepts + live coding · 30 min practice

| # | What we cover | Time |
|---|---|---|
| 1 | **Label vs position — `loc` and `iloc`** | 25 min |
| 2 | Boolean filtering | 25 min |
| 3 | Handy filters — `isin`, `between`, `query` | 15 min |
| 4 | **Setting the index, and the trap it creates** | 15 min |
| 5 | Mini build: five questions, five filters | 5 min |
| 6 | **Practice notebook (separate file)** | 30 min |

> **One sentence to carry through the whole session.**
> `loc` speaks **labels**. `iloc` speaks **positions**. Every mistake today comes from asking
> one of them a question in the other's language — and because a DataFrame's labels often
> happen to *be* 0, 1, 2, the two agree just often enough to lull you into not thinking
> about it.

In [3]:
import pandas as pd
import numpy as np

df = pd.DataFrame({
    "name":   ["Ravi", "Sara", "Amit", "Neha", "Kiran"],
    "city":   ["Pune", "Mumbai", "Delhi", "Pune", "Delhi"],
    "python": [88, 91, 45, 67, 72],
    "stats":  [71, 84, 38, 73, 69],
})
df

,name,city,python,stats
0,Ravi,Pune,88,71
1,Sara,Mumbai,91,84
2,Amit,Delhi,45,38
3,Neha,Pune,67,73
4,Kiran,Delhi,72,69


---
# 1. Label vs position

## 1.1 The distinction

In [ ]:
# On a fresh DataFrame the labels ARE 0,1,2,3,4 - same as the positions.
# So these two agree...
print("loc [1, 'name'] :", df.loc[1, "name"])
print("iloc[1, 0]      :", df.iloc[1, 0])

# ...and that is exactly why the difference is easy to miss.

In [ ]:
# Where they part company
print("df.iloc[-1]['name'] :", df.iloc[-1]["name"])     # the last row

try:
    df.loc[-1]
except KeyError:
    print("df.loc[-1]          -> KeyError: there is no LABEL -1")

| | `loc` | `iloc` |
|---|---|---|
| speaks | **labels** | **positions** |
| `[2]` means | the row *labelled* 2 | the *3rd* row |
| `[-1]` | `KeyError` | the last row |
| columns by | name | number |
| accepts a mask | yes | no |

**If you are counting from the top, you want `iloc`. If you are naming something, you want
`loc`.**

## 1.2 The slicing difference

In [1]:
print("df.loc[1:3]  ->", df.loc[1:3, "name"].tolist(), " <- 3 rows, INCLUSIVE")
print("df.iloc[1:3] ->", df.iloc[1:3]["name"].tolist(), " <- 2 rows, EXCLUSIVE")

NameError: name 'df' is not defined

**Why this is not really an inconsistency.**

`iloc` slices by position, so it follows Python's rule from Day 4 — the stop is excluded,
exactly like a list or `range()`.

`loc` slices by label, and with labels there is no "next one" to stop before. If you ask for
*"Ravi to Neha"* you clearly mean Neha included. Once you think of `loc` as **naming a first
and last row** rather than counting, the behaviour is the obvious one.

In [ ]:
# With a string index, loc's inclusivity feels natural
named = df.set_index("name")
print(named.loc["Sara":"Neha"].index.tolist())
# Sara TO Neha - of course Neha is included

---
# 2. Boolean filtering

## 2.1 A mask is a boolean Series

In [4]:
mask = df["python"] > 70
print(mask)
print()
print("type:", type(mask).__name__)

0     True
1     True
2    False
3    False
4     True
Name: python, dtype: bool

type: Series


In [11]:
# Pass the mask to df to keep only the True rows
df[mask]

# Or, in one line, which is what you will usually write:
df[(df["python"] > 70) | (df["stats"] > 70)]["city"].tolist()

['Pune', 'Mumbai', 'Pune', 'Delhi']

This is Day 12's array masking with labels attached.

## 2.2 Combining conditions

In [9]:
print("both subjects >70 :", df[(df["python"] > 70) & (df["stats"] > 70)]["name"].tolist())
print("Pune OR high mark :", df[(df["city"] == "Pune") | (df["python"] > 85)]["name"].tolist())
print("not from Pune     :", df[~(df["city"] == "Pune")]["name"].tolist())

both subjects >70 : ['Ravi', 'Sara']
Pune OR high mark : ['Ravi', 'Sara', 'Neha']
not from Pune     : ['Sara', 'Amit', 'Kiran']


In [ ]:
# The Day 13 rule, for the third time
try:
    df[(df["python"] > 70) and (df["stats"] > 70)]
except ValueError as e:
    print("using 'and' ->", str(e)[:60])
    print("   Use  &  for and,  |  for or,  ~  for not.")
    print("   And bracket every condition - & binds tighter than >.")

## 2.3 ⚠️ Changing values — and the assignment that vanishes

In [15]:
import warnings

d = df.copy()
print("before:", d.loc[2, "python"])

with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    d[d["python"] < 50]["python"] = 50        # TWO sets of brackets

print("after :", d.loc[2, "python"], "  <- unchanged!")

# pandas built a temporary COPY of the filtered rows, wrote 50 into the copy,
# and threw the copy away. Modern pandas warns about this; older versions
# sometimes worked and sometimes did not, which was worse.

before: 45
after : 45   <- unchanged!


In [13]:
df

,name,city,python,stats
0,Ravi,Pune,88,71
1,Sara,Mumbai,91,84
2,Amit,Delhi,45,38
3,Neha,Pune,67,73
4,Kiran,Delhi,72,69


In [20]:
d = df.copy()
d.loc[d["python"] > 0, "python"] = 50        # ONE .loc, rows AND column

print("with .loc:", d.loc[2, "python"], "  <- changed, as intended")
d

with .loc: 50   <- changed, as intended


,name,city,python,stats
0,Ravi,Pune,50,71
1,Sara,Mumbai,50,84
2,Amit,Delhi,50,38
3,Neha,Pune,50,73
4,Kiran,Delhi,50,69


> **The rule:** to change data, use one `.loc[rows, column]` — never two square brackets in
> a row.
>
> **Where you have met this before.** This is the copy-versus-view question from Day 12 in a
> new costume. There, a NumPy slice was a *view* and surprised you by changing the original.
> Here, a filtered DataFrame is a *copy* and surprises you by **not** changing it. Same
> underlying question, opposite symptom.

---
# 3. Handy filters

In [21]:
# isin - instead of a chain of == comparisons
print(df[df["city"].isin(["Pune", "Delhi"])]["name"].tolist())

# between - INCLUSIVE at both ends
print(df[df["python"].between(60, 90)]["name"].tolist())

# query - a string expression, where 'and' IS allowed
print(df.query("python > 70 and city == 'Delhi'")["name"].tolist())

['Ravi', 'Amit', 'Neha', 'Kiran']
['Ravi', 'Neha', 'Kiran']
['Kiran']


In [27]:
# Sorting, and top-n
print(df.sort_values("python", ascending=False)[["name", "python"]])
print()
print("top 2, whole rows kept:")
print(df.nsmallest(3, "stats")[["name", "city", "python","stats"]])

    name  python
1   Sara      91
0   Ravi      88
4  Kiran      72
3   Neha      67
2   Amit      45

top 2, whole rows kept:
    name   city  python  stats
2   Amit  Delhi      45     38
4  Kiran  Delhi      72     69
0   Ravi   Pune      88     71


### ⚠️ `between` includes both ends

Unlike almost every other range in Python — `range()`, slicing, `np.arange` — both endpoints
are kept by default. If a boundary value matters (a pass mark, an age limit), check which
side you meant.

In [ ]:
p = df["python"]
print("between(45, 88)                    :", df[p.between(45, 88)]["name"].tolist())
print("between(45, 88, inclusive='neither'):", df[p.between(45, 88, inclusive="neither")]["name"].tolist())
# The second drops Amit (45) and Ravi (88) - the exact boundary values.

---
# 4. Setting the index — and the trap

## 4.1 ⚠️ Filtering breaks the label/position match

In [28]:
top = df[df["python"] > 70]
print(top[["name", "python"]])
print()
print("index after filtering:", top.index.tolist())
print("  labels    0, 1, 4")
print("  positions 0, 1, 2   <- they no longer match")

    name  python
0   Ravi      88
1   Sara      91
4  Kiran      72

index after filtering: [0, 1, 4]
  labels    0, 1, 4
  positions 0, 1, 2   <- they no longer match


In [29]:
print("top.iloc[2]['name'] :", top.iloc[2]["name"], "  (the 3rd row)")

try:
    top.loc[2]
except KeyError:
    print("top.loc[2]          -> KeyError: label 2 was Amit, who was filtered out")

print("top.loc[4]['name']  :", top.loc[4]["name"], "  (label 4 still exists)")

top.iloc[2]['name'] : Kiran   (the 3rd row)
top.loc[2]          -> KeyError: label 2 was Amit, who was filtered out
top.loc[4]['name']  : Kiran   (label 4 still exists)


**This is the single most valuable thing in today's session.** `loc[2]` fails on a row you
can see on screen, because you asked for a *label* that no longer exists.

In [30]:
top = top.reset_index(drop=True)
print(top[["name", "python"]])
print("index now:", top.index.tolist(), " <- labels match positions again")

    name  python
0   Ravi      88
1   Sara      91
2  Kiran      72
index now: [0, 1, 2]  <- labels match positions again


In [34]:
# Without drop=True, the old labels are KEPT as a new column
again = df[df["python"] > 70].reset_index(drop = True)
print(again.columns.tolist())
again
# ['index', 'name', 'city', 'python', 'stats']  <- a stray 'index' column

['name', 'city', 'python', 'stats']


,name,city,python,stats
0,Ravi,Pune,88,71
1,Sara,Mumbai,91,84
2,Kiran,Delhi,72,69


## 4.2 `set_index` — giving rows meaningful names

In [35]:
named = df.set_index("name")
named

,city,python,stats
name,,,
Ravi,Pune,88,71
Sara,Mumbai,91,84
Amit,Delhi,45,38
Neha,Pune,67,73
Kiran,Delhi,72,69


In [41]:
print(named.loc["Sara"])
print()
print("one value      :", named.loc["Sara", "python"])
print("a label slice  :", named.loc["Ravi":"Amit"].index.tolist())
print("iloc unchanged :", named.iloc[0:2], " <- iloc is ALWAYS positions")

city      Mumbai
python        91
stats         84
Name: Sara, dtype: object

one value      : 91
a label slice  : ['Ravi', 'Sara', 'Amit']
iloc unchanged :         city  python  stats
name                       
Ravi    Pune      88     71
Sara  Mumbai      91     84  <- iloc is ALWAYS positions


**When to set an index — and when not to bother.**

Set one when the rows have a natural identifier you will look things up by: a student ID, a
product code, a date. Leave the default `0, 1, 2` when they do not — a made-up index adds
nothing. Dates are the case where it pays off most, and Module 4's time series work assumes
it.

---
# 5. Putting it together — five questions, five filters

In [ ]:
df = pd.DataFrame({
    "name":   ["Ravi", "Sara", "Amit", "Neha", "Kiran"],
    "city":   ["Pune", "Mumbai", "Delhi", "Pune", "Delhi"],
    "python": [88, 91, 45, 67, 72],
    "stats":  [71, 84, 38, 73, 69],
})

# 1. Who scored above 70 in python?
top = df[df["python"] > 70]
print("1.", top["name"].tolist())

# 2. Who is strong in BOTH subjects?
both = df[(df["python"] > 70) & (df["stats"] > 70)]
print("2.", both["name"].tolist())

# 3. Who is from Pune or Delhi?
local = df[df["city"].isin(["Pune", "Delhi"])]
print("3.", local["name"].tolist())

# 4. The three highest python marks, whole rows kept
print("4.", df.nlargest(3, "python")["name"].tolist())

# 5. Give everyone below 50 a resit mark of 50
df.loc[df["python"] < 50, "python"] = 50       # ONE loc
print("5. Amit is now", df.loc[2, "python"])

In [ ]:
# Tidy the filtered result before reporting it
top = top.reset_index(drop=True)
print(top[["name", "python"]])

# And note: filtering returned a COPY, so changing df did not change top.
# That is the reverse of NumPy's view behaviour from Day 12.

---
# 6. Recap — the twelve things to remember

1. `loc` speaks **labels**. `iloc` speaks **positions**.
2. On a fresh DataFrame they agree — that is the trap.
3. `loc` slicing **includes** the stop; `iloc` excludes it.
4. `df.loc[-1]` is a `KeyError`; `df.iloc[-1]` is the last row.
5. A mask is a boolean Series: `df[df["col"] > 70]`.
6. Use `&` `|` `~` and bracket every condition. Never `and` / `or`.
7. To **assign**, use one `df.loc[rows, col] = value`.
8. `df[mask]["col"] = v` writes to a copy and vanishes.
9. Filtering keeps the **original** labels — gaps appear.
10. So `loc[2]` can fail on a row you can see. Use `iloc`, or reset.
11. `reset_index(drop=True)` makes labels match positions again.
12. `isin`, `between` (inclusive!), `query`, `nlargest`, `sort_values`.

---

### 📝 Now open **`Day16_Practice_Questions.ipynb`** for the 30-minute practice session.

### Homework
- Filter a dataset, then show `loc` and `iloc` disagreeing on the same number.
- Write five questions about a CSV and answer each with one filter.
- Set a meaningful index on a dataset and look up three rows by name.

### Next class — Topic 1.16: GroupBy & aggregation
Split the table into groups, compute a summary for each, and put the answers back together.

---
*Slides & notebooks by Srinivasa Sai Chava · Boston University*